# Water Bodies & Flood Mapping
**NB 07** | Bangladesh floodplain | Sentinel-2 | Prithvi flood + MNDWI

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json
import pathlib

import numpy as np

import pygeovision as pgv

client = pgv.PyGeoVision()
print(client)

from pygeovision.models.foundation.prithvi import PrithviTasks


In [ ]:
BBOX       = (89.0, 23.5, 89.5, 24.0)
DATE_RANGE = ('2024-07-01', '2024-10-31')
PROVIDERS  = ['planetary_computer']

DATA_DIR = pathlib.Path('./results/notebook')
DATA_DIR.mkdir(parents=True, exist_ok=True)

results = client.search(
    bbox            = BBOX,
    date_range      = DATE_RANGE,
    providers       = PROVIDERS,
    cloud_cover_max = 25,
)
print(f"Found {len(results)} scenes")
for r in results[:5]:
    print(f"  {r.provider:<22} {r.datetime[:10]}  cloud={r.cloud_cover:.1f}%  {r.id[:40]}")


In [ ]:
BANDS = ['B03', 'B08', 'B11']

downloads = client.download(
    results[:1],
    output_dir   = str(DATA_DIR),
    bands        = BANDS,
    post_process = ['reproject:EPSG:32646', 'cog'],
)
scene_path = downloads[0].path if downloads and downloads[0].success else None
scl_cands  = list(DATA_DIR.rglob('*SCL*.tif')) + list(DATA_DIR.rglob('*scl*.tif'))
scl_path   = str(scl_cands[0]) if scl_cands else None
if scene_path:
    d = downloads[0]
    print(f"Downloaded   : {scene_path}")
    print(f"Size         : {d.bytes_downloaded/1024/1024:.1f} MB")
    print(f"SCL mask     : {scl_path}")
else:
    print("Download failed or scene unavailable — check provider availability")


In [ ]:
if scene_path:
    raw_report = client.validator.validate(
        str(scene_path),
        required_bands = 3,
        value_range    = (0, 65535),
    )
    print("Raw data validation:")
    print(raw_report.summary())
    if not raw_report.passed:
        print("WARNING: Issues found — preprocessing will auto-fix")


In [ ]:
PREPROCESSED = DATA_DIR / 'bangladesh_preprocessed.tif'

if scene_path:
    ready = client.prepare_for_ai(
        str(scene_path),
        stack_bands  = BANDS,
        bbox         = BBOX,
        scl_path         = scl_path,
        scl_keep_classes = [4,5,6],
        normalise    = 'scale_factor',
        scale_factor = 10000.0,
        model_type   = 'segmentation',
        output_path  = str(PREPROCESSED),
    )
    print("Preprocessing steps :", ready['steps'])
    print("Output shape (C,H,W):", ready['shape'])
    print("Resolution          :", ready['resolution_m'], "m")
    print("Validation          :", "PASSED" if ready['report'] and ready['report'].passed else "FIXED (auto)")
    arr = ready['array']
    print(f"Value range         : {arr.min():.4f} → {arr.max():.4f}")
    print(f"NaN count           : {np.isnan(arr).sum()}")
else:
    print("No scene available — cannot preprocess")
    ready = None


In [ ]:
# MNDWI water index on preprocessed data (band map: green=1, nir=2, swir1=3)
if PREPROCESSED.exists():
    idx_dir = DATA_DIR / 'indices'
    idx_dir.mkdir(exist_ok=True)
    mndwi = client.indices.mndwi(str(PREPROCESSED), green_band=1, swir1_band=3,
        output_path=str(idx_dir/'mndwi.tif'))
    ndwi  = client.indices.ndwi(str(PREPROCESSED),  green_band=1, nir_band=2,
        output_path=str(idx_dir/'ndwi.tif'))
    import rasterio
    for name, p in [('MNDWI', mndwi), ('NDWI', ndwi)]:
        with rasterio.open(p) as src:
            a = src.read(1)
        print(f"{name}: mean={a.mean():.3f}  water(>0)={( a>0).mean()*100:.1f}%")


In [ ]:
# Prithvi flood detection on preprocessed + full-band scene
FLOOD_MAP  = DATA_DIR / 'flood_map.tif'
# Prithvi needs 6-band HLS: download the full Sentinel-2 stack and re-preprocess
full_bands = ['B02','B03','B04','B08','B11','B12']
full_dl = client.download(results[:1], output_dir=str(DATA_DIR/'full'),
    bands=full_bands, post_process=['reproject:EPSG:32646','cog'])
if full_dl and full_dl[0].success:
    full_ready = client.prepare_for_ai(str(full_dl[0].path), stack_bands=full_bands,
        bbox=BBOX, normalise='scale_factor', scale_factor=10000.0,
        model_type='segmentation', output_path=str(DATA_DIR/'full_preprocessed.tif'))
    tasks = PrithviTasks('prithvi_eo_2_0')
    tasks.flood_detection(str(DATA_DIR/'full_preprocessed.tif'),
        source='sentinel2', output_path=str(FLOOD_MAP))
    print(f"Flood map: {FLOOD_MAP}")


In [ ]:
if PREDICTION.exists():
    pred_report = client.validator.validate(str(PREDICTION))
    print("Prediction validation:", "PASSED" if pred_report.passed else f"FIXED — {pred_report.errors}")
    print(pred_report.summary())


In [ ]:
SIEVED  = DATA_DIR / 'sieved.tif'
VECTOR  = DATA_DIR / 'output.geojson'
COG_OUT = DATA_DIR / 'output_cog.tif'

if PREDICTION.exists():
    client.postprocess.sieve_filter(str(PREDICTION), min_pixels=15, output_path=str(SIEVED))
    client.postprocess.vectorise(str(SIEVED), str(VECTOR),
        target_class=1, min_area_m2=50.0, simplify_tolerance=0.5)
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    with open(VECTOR) as f: gj = json.load(f)
    print(f"Polygons detected : {len(gj['features'])}")
    for cls, info in sorted(stats.items()):
        print(f"  Class {cls}: {info['pixels']:>8,} px | {info['area_ha']:>8.1f} ha | {info['pct']:>6.1f}%")
    print(f"COG saved: {COG_OUT}")


In [ ]:
# Zonal flood statistics
VECTOR = DATA_DIR / 'output.geojson'
if VECTOR.exists() and (DATA_DIR/'indices'/'mndwi.tif').exists():
    zonal = client.postprocess.zonal_statistics(
        str(DATA_DIR/'indices'/'mndwi.tif'), str(VECTOR),
        output_path=str(DATA_DIR/'flood_mndwi_stats.geojson'),
        stats=['mean','max','count'],
    )


**NB07 — Flood Mapping** complete.
- Study area: Bangladesh floodplain
- Sensor: Sentinel-2
- Model: Prithvi-EO-2.0 + MNDWI
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG